# Qwen3-TTS dialogue renderer v4 (jelly-jlpt)

Renders the **lesson-dialogue characters** (`tools/audio/cast.json`) from saved voice references. Exam / listening tracks (narrator, man / woman archetypes) still use `render-notebook-v2.ipynb`.

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom (section 4 is collapsed and optional: skip it unless a character is new or its voice changed).

1. **Setup**: install, models, helpers, Drive (clips + refs live in `MyDrive/jelly-audio-render`, so a disconnect loses nothing).
2. **Voice references**: upload `tools/audio/refs/*.wav + *.json` once. They stay on Drive; later sessions skip this.
3. **Render input**: upload `tools/audio/render-input.json` (from `node tools/export-tracks.js`).
4. **(Optional) New character / changed voice**: design, listen, save, download the new reference. Commit it to `tools/audio/refs/`.
5. **Render**: all remaining clips, or only the characters you list.
6. **Manifest + download**: verifies, writes `manifest.json`, downloads `jelly-audio-new.zip`.

**Then, in the repo:** unzip (e.g. `.scratch/render/`), copy the mp3s into `audio/`, `node tools/build-audio-manifest.js .scratch/render`, `node tools/build-sw.js`, `node .claude/hooks/run-tests.js`.

One clip that drifted to another voice: give that line `take: 2` in `data/n5/listening.js`, re-export, render again (see README).

## 1. Setup

In [ ]:
#@title Install, models, helpers, Drive { display-mode: "form" }
!pip install -q -U qwen-tts soundfile tqdm

import os, json, hashlib, zlib, subprocess, zipfile
import torch, numpy as np, soundfile as sf
from IPython.display import Audio, display
from google.colab import files as files_mod
from qwen_tts import Qwen3TTSModel
from tqdm.auto import tqdm
print(torch.cuda.get_device_name(0))

def load(name):  # T4: float16 (no bfloat16), no flash-attn
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.float16)

_models = {}
def model(kind):  # loaded on first use: "design" only while tuning a new voice, "base" clones every line
    ids = {"design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
           "base":   "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}
    if kind not in _models: _models[kind] = load(ids[kind])
    return _models[kind]

def design_ref(voice, text, seed):
    """VoiceDesign render of the reference text: the voice to lock. Not reproducible across sessions, so it is saved."""
    torch.manual_seed(seed)
    w, sr = model("design").generate_voice_design(text=text, language="Japanese", instruct=voice)
    return np.asarray(w[0], dtype=np.float32), sr

def cloner(ref):
    """speak(text, seed) -> (wave, sr): the Base model cloning one fixed reference clip."""
    prompt = model("base").create_voice_clone_prompt(ref_audio=(ref["wave"], ref["sr"]), ref_text=ref["text"])
    def speak(text, s):
        torch.manual_seed(s)
        w, sr2 = model("base").generate_voice_clone(text=text, language="Japanese", voice_clone_prompt=prompt)
        return np.asarray(w[0], dtype=np.float32), sr2
    return speak

def loudness(w, target_db):
    """Scale a line so its speech (silence ignored) sits at target_db RMS; never clip."""
    active = w[np.abs(w) > 0.02 * np.abs(w).max()]
    g = 10 ** (target_db / 20) / (np.sqrt(np.mean(active ** 2)) + 1e-9)
    w = w * g
    peak = np.abs(w).max()
    return w * (0.97 / peak) if peak > 0.97 else w

# the values approved for v2 / v3
SEED      = 1234                       # base of each clip's seed: SEED + crc32(clipKey) % 100000
TARGET_DB = -20
GAIN_DB   = {"F": 1, "M": 0, "N": 0}
clip_seed = lambda key: SEED + zlib.crc32(key.encode("utf-8")) % 100000

def play(w, sr, role):
    display(Audio(loudness(w, TARGET_DB + GAIN_DB.get(role, 0)), rate=sr))

def clip_key(role, arch, say, take=None):     # same rule as tools/build-audio-manifest.js; arch '<id>@<rev>' = a revised voice
    return role + "|" + ("" if role == "N" else arch) + "|" + say + ("#" + str(take) if take and take > 1 else "")  # '#<take>' = a re-render

def clip_name(key):
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:12] + ".mp3"

# output folder on Google Drive, so a disconnect resumes
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/jelly-audio-render"
except Exception as e:
    print("Drive not mounted (", e, "): using /content, NOT resumable and saved references are lost on disconnect")
    OUT = "/content/jelly-audio-render"
REFS = os.path.join(OUT, "refs")       # voice references: <id>@<rev>.wav + .json
os.makedirs(REFS, exist_ok=True)
for d in (OUT, REFS):                  # leftovers of an interrupted write
    for f in os.listdir(d):
        if f.endswith(".tmp"): os.remove(os.path.join(d, f))

ref_file = lambda rid, ext: os.path.join(REFS, rid + ext)
on_disk  = lambda: {f for f in os.listdir(OUT) if f.endswith(".mp3")}
saved_refs = lambda: sorted(f[:-5] for f in os.listdir(REFS) if f.endswith(".json"))

def write_mp3(w, sr, role, name):      # loudness + MP3, written to a temp file and renamed (no half clips)
    sf.write("/content/_tmp.wav", loudness(w, TARGET_DB + GAIN_DB.get(role, 0)), sr)
    tmp = os.path.join(OUT, name + ".tmp")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/_tmp.wav", "-ac", "1", "-ar", "24000", "-b:a", "48k", "-f", "mp3", tmp], check=True)
    os.replace(tmp, os.path.join(OUT, name))

def load_ref(rid):
    meta = json.load(open(ref_file(rid, ".json"), encoding="utf-8"))
    w, sr = sf.read(ref_file(rid, ".wav"), dtype="float32")
    return {**meta, "wave": w, "sr": sr}

print("output:", OUT, "| saved references:", saved_refs() or "none")

## 2. Voice references (once)
Select every `.wav` and `.json` from `tools/audio/refs/`. They are copied to Drive and stay there: when Drive already holds references this cell does nothing (set `RE_UPLOAD = True` to upload again, e.g. after committing a new one).

In [ ]:
RE_UPLOAD = False

if saved_refs() and not RE_UPLOAD:
    print("Already on Drive, nothing to upload:", ", ".join(saved_refs()))
else:
    print("Select all of tools/audio/refs/*.wav and *.json")
    for name, data in files_mod.upload().items():
        open(ref_file(os.path.splitext(os.path.basename(name))[0], os.path.splitext(name)[1]), "wb").write(data)
    print("Saved references:", ", ".join(saved_refs()))

## 3. Render input
Upload `tools/audio/render-input.json`. Prints each character: lines, clips, already rendered, still to render, saved reference.

In [ ]:
print("Upload tools/audio/render-input.json")
up = files_mod.upload()
DATA = json.loads(next(iter(up.values())).decode("utf-8"))
TRACKS, CHARS, HAVE = DATA["tracks"], DATA["characters"], set(DATA.get("have", []))

def arch_of(track, line):                    # a dialogue line names its character; the rest use the track's man / woman archetype
    return line.get("arch") or {"M": track["man"], "F": track["woman"]}.get(line["role"]) or ""

CLIPS, LINES = {}, {}                         # clip name -> (role, arch, say, key); arch -> number of lines
for t in TRACKS:
    for l in t["lines"]:
        a = arch_of(t, l); k = clip_key(l["role"], a, l["say"], l.get("take"))
        CLIPS.setdefault(clip_name(k), (l["role"], a, l["say"], k))
        LINES[a] = LINES.get(a, 0) + 1

rev_of   = lambda c: CHARS[c].get("rev", 1)
arch_tag = lambda c: c if rev_of(c) == 1 else f"{c}@{rev_of(c)}"   # the clipKey archetype (charArch in build-audio-manifest.js)
ref_id   = lambda c: f"{c}@{rev_of(c)}"                             # the saved reference's name
has_ref  = lambda c: os.path.exists(ref_file(ref_id(c), ".json"))
char_clips = lambda c: sorted(n for n, v in CLIPS.items() if v[1] == arch_tag(c))
left_of    = lambda c, done: [n for n in char_clips(c) if n not in HAVE and n not in done]

def locked(c):                       # the clone of the saved reference; refuses without one
    rid = ref_id(c)
    assert has_ref(c), f"No saved reference for {rid}: tune and save it in section 4 first."
    ref = load_ref(rid)
    assert ref["voice"] == CHARS[c]["voice"], (f"refs/{rid} was designed from another prompt than cast.json gives {c}. "
        "Paste the saved prompt into cast.json with a new rev, or tune and save again.")
    return cloner(ref)

def status():
    done = on_disk()
    print(f"{'id':9} {'name':9} {'rev':>3} {'lines':>5} {'clips':>5} {'rendered':>8} {'to render':>9}  saved reference")
    todo = []
    for c in CHARS:
        n, left = len(char_clips(c)), len(left_of(c, done))
        if left: todo.append(c)
        print(f"{c:9} {CHARS[c]['name']:9} {rev_of(c):>3} {LINES.get(arch_tag(c), 0):>5} {n:>5} {n - left:>8} {left:>9}  {ref_id(c) if has_ref(c) else '- MISSING'}")
    tags = {arch_tag(c) for c in CHARS}
    other = [n for n, v in CLIPS.items() if v[1] not in tags and n not in HAVE and n not in done]
    print("\nCharacters with clips to render:", ", ".join(todo) or "none")
    if other: print(len(other), "exam / listening clips (narrator, man / woman archetypes) are also missing: render them with render-notebook-v2.ipynb")
status()

## 4. (Optional) New character or changed voice
**Skip this unless a character has no saved reference (table above says MISSING) or you are changing a voice.** This section is collapsed by default; with `NEW_CHARACTER = ""` both cells do nothing, so Run all is safe.

Tune: designs a voice from `VOICE` + `VOICE_SEED` and plays 3 of the character's lines cloned from it. Change the seed (or prompt) and re-run until it sounds right. Save: writes `refs/<id>@<rev>` to Drive and downloads `refs-new.zip`: put its files in `tools/audio/refs/` and commit them. A changed prompt is saved as the next rev and the cell prints the `"rev"` and `"voice"` lines to paste into the character's entry in `tools/audio/cast.json`; then re-run `node tools/export-tracks.js` and redo section 3.

In [ ]:
NEW_CHARACTER = ""                      # a character id from the table above; empty = skip this section
VOICE_SEED    = 1234                    # try 1, 2, 3, ...: other voices from the same prompt
VOICE         = ""                      # empty = the prompt from cast.json; edit to try another
REF_TEXT      = "おはようございます。今日は天気がいいので、駅まで歩いて行きます。いいえ、違いますよ。"   # what the reference says (saved with it)

def samples(c):                         # a short line, a long line, one starting with ……
    names = sorted(char_clips(c), key=lambda n: len(CLIPS[n][2]))
    pick = [names[0], names[-1]] + [n for n in names[1:-1] if CLIPS[n][2].startswith("……")][:1]
    return list(dict.fromkeys(pick))

if not NEW_CHARACTER:
    print("Skipped (NEW_CHARACTER is empty).")
else:
    assert NEW_CHARACTER in CHARS, f"{NEW_CHARACTER!r} is not a character: {', '.join(CHARS)}"
    ROLE = CHARS[NEW_CHARACTER]["gender"]
    VOICE = VOICE or CHARS[NEW_CHARACTER]["voice"]
    w, sr = design_ref(VOICE, REF_TEXT, VOICE_SEED)
    CUR = {"voice": VOICE, "seed": VOICE_SEED, "text": REF_TEXT, "wave": w, "sr": sr}
    print("Prompt:", VOICE, "\nSeed:", VOICE_SEED, "(not saved yet)\nReference:", REF_TEXT); play(w, sr, ROLE)
    speak = cloner(CUR)
    for n in samples(NEW_CHARACTER):    # each with the seed the render will use for that clip
        role, arch, say, key = CLIPS[n]
        w2, sr2 = speak(say, clip_seed(key))
        print(say); play(w2, sr2, role)

In [ ]:
SAVE_REFERENCE = False   # set True and run once the samples above sound right
OVERWRITE      = False   # True replaces an existing saved reference: only if none of its clips are rendered yet (else they no longer match)

if SAVE_REFERENCE and NEW_CHARACTER:
    changed = CUR["voice"] != CHARS[NEW_CHARACTER]["voice"]
    rid = f"{NEW_CHARACTER}@{rev_of(NEW_CHARACTER) + 1}" if changed else ref_id(NEW_CHARACTER)
    if os.path.exists(ref_file(rid, ".json")) and not OVERWRITE:
        print(rid, "is already saved: nothing written (set OVERWRITE = True to replace it)")
    else:
        sf.write(ref_file(rid, ".wav.tmp"), CUR["wave"], CUR["sr"], format="WAV", subtype="FLOAT")   # lossless
        os.replace(ref_file(rid, ".wav.tmp"), ref_file(rid, ".wav"))
        json.dump({"id": NEW_CHARACTER, "rev": int(rid.split("@")[1]), "voice": CUR["voice"], "seed": CUR["seed"], "text": CUR["text"]},
                  open(ref_file(rid, ".json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        print("Saved", rid, "to Drive")
        with zipfile.ZipFile("/content/refs-new.zip", "w") as z:
            for ext in (".wav", ".json"): z.write(ref_file(rid, ext), rid + ext)
        files_mod.download("/content/refs-new.zip")
        print("Downloaded refs-new.zip: copy its files into tools/audio/refs/ and commit.")
        if changed:
            print(f"\nThe prompt changed. In tools/audio/cast.json, in the \"{NEW_CHARACTER}\" entry, set these two fields:\n")
            print(f'"rev": {rid.split("@")[1]},')
            print(f'"voice": {json.dumps(CUR["voice"], ensure_ascii=False)}')
            print("\nThen run node tools/export-tracks.js and redo section 3 with the new render-input.json.")
else:
    print("Nothing saved.")

## 5. Render
Set `RENDER` to the characters to render, or leave it empty to render every character that still has clips left. Safe to stop and re-run: finished clips are skipped, each clip is written atomically. A character without a saved reference is skipped (all mode) or refused (listed mode).

In [ ]:
RENDER = []        # e.g. ["yor", "sanji"]; empty = all remaining

bad = [c for c in RENDER if c not in CHARS]
assert not bad, f"not characters: {bad}. Choose from: {', '.join(CHARS)}"
want = RENDER or [c for c in CHARS if left_of(c, on_disk())]
noref = [c for c in want if not has_ref(c)]
assert not (RENDER and noref), f"No saved reference for {noref}: upload it (section 2) or tune and save it (section 4)."
if noref: print("SKIPPED, no saved reference (section 4):", ", ".join(noref), "\n")

total = 0
for c in [c for c in want if c not in noref]:
    todo = left_of(c, on_disk())
    if not todo: continue
    speak = locked(c)
    for n in tqdm(todo, desc=ref_id(c)):
        role, arch, say, key = CLIPS[n]
        w, sr = speak(say, clip_seed(key))
        write_mp3(w, sr, role, n)
    total += len(todo)
print(total, "clips rendered\n")
status()

## 6. Manifest and download
Needs every clip of every track (dialogue and exam) in `audio/` or the output folder. Writes `manifest.json` and downloads `jelly-audio-new.zip` (new mp3s + `manifest.json`).

In [ ]:
status()
missing = [n for n in CLIPS if n not in HAVE and n not in on_disk()]
assert not missing, f"{len(missing)} clips missing: render the characters listed above (exam tracks: v2 notebook)"
manifest = {"version": 1, "tracks": {t["id"]: [clip_name(clip_key(l["role"], arch_of(t, l), l["say"], l.get("take"))) for l in t["lines"]] for t in TRACKS}}
assert all(len(manifest["tracks"][t["id"]]) == len(t["lines"]) for t in TRACKS)
tmp = os.path.join(OUT, "manifest.json.tmp")
json.dump(manifest, open(tmp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
os.replace(tmp, os.path.join(OUT, "manifest.json"))

NEW = sorted(n for n in CLIPS if n not in HAVE)
ZIP = "/content/jelly-audio-new.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_STORED) as z:   # mp3 does not compress
    for n in NEW: z.write(os.path.join(OUT, n), n)
    z.write(os.path.join(OUT, "manifest.json"), "manifest.json")
print(f"OK: {len(CLIPS)} clips accounted for, {len(NEW)} new, {len(manifest['tracks'])} tracks in manifest.json")
print(f"zip: {os.path.getsize(ZIP) / 1e6:.1f} MB")
files_mod.download(ZIP)